In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

FILES = {
    'DoS':        'DoS.csv',
    'DDoS':       'DDoS.csv',
    'Port Scan':  'PortScan.csv',
    'Brute Force':'BruteForce.csv',
}

FEATURES = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets',
    'Total Backward Packets', 'Total Length of Fwd Packets',
    'Total Length of Bwd Packets', 'Fwd Packet Length Max',
    'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std',
    'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean',
    'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total',
    'Bwd IAT Mean', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length',
    'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length',
    'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',
    'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count',
    'ACK Flag Count', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward',
    'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean',
    'Active Max', 'Active Min', 'Idle Mean',
]

In [ ]:
# Load data
print("Loading CSVs...")
dfs = {}
for label, path in FILES.items():
    dfs[label] = pd.read_csv(path, usecols=lambda c: c in FEATURES + ['Attack Type'])
    print(f"  {label}: {len(dfs[label])} rows, {dfs[label].shape[1]} cols")

In [ ]:
# 1. Descriptive statistics per class
for label, df in dfs.items():
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    display(df[FEATURES].describe().T.round(4))

In [ ]:
# 2. Feature means per class (replaces heatmap)
means = pd.DataFrame({label: df[FEATURES].mean() for label, df in dfs.items()})
means_norm = (means - means.min(axis=1).values.reshape(-1,1)) / \
             (means.max(axis=1) - means.min(axis=1) + 1e-9).values.reshape(-1,1)

print("Raw means:")
display(means.round(4))
print("\nNormalised means (0-1):")
display(means_norm.round(4))

In [ ]:
# 3. Violin plot replacement - percentile summary per feature per class (99th pct clip)
rows = []
for feat in FEATURES:
    for label, df in dfs.items():
        vals = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
        vals = vals[vals < vals.quantile(0.99)]
        q = np.percentile(vals, [0, 5, 25, 50, 75, 95, 100])
        rows.append({
            'Feature': feat, 'Class': label,
            'Min': q[0], 'P5': q[1], 'P25': q[2], 'Median': q[3],
            'P75': q[4], 'P95': q[5], 'Max': q[6],
            'Mean': vals.mean(), 'Std': vals.std()
        })

display(pd.DataFrame(rows).round(4))

In [ ]:
# 4. Box plot replacement - IQR summary per feature per class (99th pct clip)
rows = []
for feat in FEATURES:
    for label, df in dfs.items():
        vals = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
        vals = vals[vals < vals.quantile(0.99)]
        q1, median, q3 = vals.quantile([0.25, 0.50, 0.75])
        iqr = q3 - q1
        lower_fence = q1 - 1.5 * iqr
        upper_fence = q3 + 1.5 * iqr
        outlier_count = int(((vals < lower_fence) | (vals > upper_fence)).sum())
        rows.append({
            'Feature': feat, 'Class': label,
            'Q1': q1, 'Median': median, 'Q3': q3, 'IQR': iqr,
            'Lower Fence': lower_fence, 'Upper Fence': upper_fence,
            'Outlier Count': outlier_count
        })

display(pd.DataFrame(rows).round(4))

In [ ]:
# 5. Correlation matrices per class
for label, df in dfs.items():
    corr = df[FEATURES].replace([np.inf, -np.inf], np.nan).dropna().corr()
    print(f"\n{'='*60}\n  {label} - Correlation Matrix\n{'='*60}")
    display(corr.round(4))

# Highly correlated pairs per class (|r| >= 0.90)
print("\nHighly correlated pairs (|r| >= 0.90) per class:")
for label, df in dfs.items():
    corr = df[FEATURES].replace([np.inf, -np.inf], np.nan).dropna().corr()
    pairs = []
    for i in range(len(corr.columns)):
        for j in range(i+1, len(corr.columns)):
            r = corr.iloc[i, j]
            if abs(r) >= 0.90:
                pairs.append({'Class': label, 'Feature_A': corr.columns[i],
                               'Feature_B': corr.columns[j], 'r': round(r, 4)})
    pairs_df = pd.DataFrame(pairs).sort_values('r', key=abs, ascending=False)
    print(f"\n  {label} ({len(pairs_df)} pairs):")
    display(pairs_df)

In [ ]:
# 6. ANOVA F-statistics (replaces bar chart)
rows = []
for feat in FEATURES:
    groups = [df[feat].replace([np.inf, -np.inf], np.nan).dropna().values for df in dfs.values()]
    try:
        f, p = stats.f_oneway(*groups)
        f_val = f if np.isfinite(f) else 0
    except Exception:
        f_val, p = 0, 1.0
    rows.append({'Feature': feat, 'F_statistic': round(f_val, 4), 'p_value': round(p, 6)})

f_df = pd.DataFrame(rows).sort_values('F_statistic', ascending=False).reset_index(drop=True)
f_df.insert(0, 'Rank', f_df.index + 1)
display(f_df)

In [ ]:
# 7. Pairplot replacement - pairwise stats for top 5 discriminating features
top5 = f_df['Feature'].head(5).tolist()
print(f"Top 5 discriminating features: {top5}")

sample = pd.concat([
    df.sample(min(500, len(df)), random_state=42).assign(Class=label)
    for label, df in dfs.items()
]).reset_index(drop=True)
sample_clean = sample[top5 + ['Class']].replace([np.inf, -np.inf], np.nan).dropna()

print("\nPer-class descriptive summary for top 5 features:")
for feat in top5:
    print(f"\n  {feat}:")
    display(sample_clean.groupby('Class')[feat].agg(['mean','std','min','median','max']).round(4))

print("\nPairwise Pearson correlations between top 5 features per class:")
rows = []
for label, df in dfs.items():
    sub = df[top5].replace([np.inf, -np.inf], np.nan).dropna()
    for i in range(len(top5)):
        for j in range(i+1, len(top5)):
            r, p = stats.pearsonr(sub[top5[i]], sub[top5[j]])
            rows.append({'Class': label, 'Feature_A': top5[i], 'Feature_B': top5[j],
                          'Pearson_r': round(r, 4), 'p_value': round(p, 6)})

display(pd.DataFrame(rows))

In [ ]:
# 8. Flow volume profile per class (replaces bar chart)
vol_feats = [
    'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
]
rows = []
for label, df in dfs.items():
    sub = df[vol_feats].replace([np.inf, -np.inf], np.nan)
    for feat in vol_feats:
        vals = sub[feat].dropna()
        rows.append({
            'Class': label, 'Feature': feat,
            'Mean': round(vals.mean(), 4), 'Median': round(vals.median(), 4),
            'Std': round(vals.std(), 4), 'Min': round(vals.min(), 4), 'Max': round(vals.max(), 4)
        })

display(pd.DataFrame(rows))

print("\nMeans table (matches bar chart data):")
vol_means = pd.DataFrame({label: df[vol_feats].replace([np.inf, -np.inf], np.nan).mean()
                           for label, df in dfs.items()})
display(vol_means.round(4))

In [ ]:
# 9. TCP flag presence rates per class (replaces bar chart)
flag_feats = ['FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count']
rows = []
for label, df in dfs.items():
    for feat in flag_feats:
        vals = df[feat].clip(upper=1)
        rows.append({
            'Class': label, 'Flag': feat,
            'Presence Rate': round(vals.mean(), 6),
            'Total Flows': len(vals),
            'Flows with Flag': int(vals.sum())
        })

display(pd.DataFrame(rows))

print("\nFlag rates matrix (matches bar chart data):")
flag_rates = pd.DataFrame({
    label: df[flag_feats].clip(upper=1).mean()
    for label, df in dfs.items()
})
display(flag_rates.round(6))

In [ ]:
# 10. KDE replacement - distribution summary per feature per class
#     same data filtering as KDE: clip 99th pct + positive values only (log-scale compatible)
rows = []
for feat in FEATURES:
    for label, df in dfs.items():
        vals = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
        vals = vals[vals < vals.quantile(0.99)]
        vals = vals[vals > 0]
        if len(vals) < 10:
            rows.append({'Feature': feat, 'Class': label, 'N': len(vals), 'Note': 'insufficient data'})
            continue
        log_vals = np.log10(vals)
        rows.append({
            'Feature': feat, 'Class': label, 'N': len(vals),
            'Mean(log10)': round(log_vals.mean(), 4),
            'Std(log10)': round(log_vals.std(), 4),
            'P5(log10)': round(np.percentile(log_vals, 5), 4),
            'P25(log10)': round(np.percentile(log_vals, 25), 4),
            'Median(log10)': round(np.percentile(log_vals, 50), 4),
            'P75(log10)': round(np.percentile(log_vals, 75), 4),
            'P95(log10)': round(np.percentile(log_vals, 95), 4),
            'Mean(raw)': round(vals.mean(), 4),
            'Median(raw)': round(vals.median(), 4),
            'Note': ''
        })

display(pd.DataFrame(rows))